In [1]:
from functools import wraps
import json
import time
from typing import Any, Callable, Dict, List, Optional, ParamSpec, TypeVar
from openai import OpenAI
from dotenv import load_dotenv
import os

P = ParamSpec("P")
T = TypeVar("T")

def retry(N) -> Callable[[Callable[P, Optional[T]]], Callable[P, Optional[T]]]:
    def retrier(func: Callable[P, Optional[T]]) -> Callable[P, Optional[T]]:
        @wraps(func)
        def wrapper(*args: P.args, **kwargs: P.kwargs) -> Optional[T]:
            for i in range(N):
                try:
                    result = func(*args, **kwargs)
                    if result is not None:
                        return result
                    print("API returned None.")
                except Exception as e:
                    pass
                    print(f"API failed: {e}.")
                print(f"Retrying {i+2}/{N} time.")
                time.sleep(2)
            print(f"API failed after {N} retries.")
            return None
        return wrapper
    return retrier


class Model():
    def __init__(self, client: OpenAI, model: str):
        self.client = client
        self.model = model


class Test():
    def __init__(self, test_prompt: str, validation_prompt: str):
        self.test_prompt = test_prompt
        self.validation_prompt = validation_prompt
        
    @retry(3)
    def run(self, test_model: Model) -> str | None:
        response = test_model.client.chat.completions.create(
            model=test_model.model,
            messages=[
                {
                    "role": "user",
                    "content": self.test_prompt
                }
            ],
            max_tokens=1024,
            timeout=60.0
        )
        return response.choices[0].message.content
    
    @retry(3)
    def validate(self, validation_model: Model, response_first: str, response_second: str) -> tuple[int, int, str] | None:
        def evaluate(score_first: int, score_second: int, comments: str) -> tuple[int, int, str] | None:
            '''
                Evaluate two models
            '''
            if score_first is not None and score_second is not None and comments:
                return (score_first, score_second, comments)
            return None
            
        tools = [
            {
                "type": "function",
                "function": {
                    "name": "evaluate",
                    "description": "Evaluate two models based on the evaluation criteria",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "score_first": {
                                "type": "integer",
                                "description": "Score for the first model"
                            },
                            "score_second": {
                                "type": "integer",
                                "description": "Score for the second model"
                            },
                            "comments": {
                                "type": "string",
                                "description": "Comments about the evaluation of the models"
                            }
                        },
                        "required": ["score_first", "score_second", "comments"]
                    }
                }
            }
        ]
        
        prompt = f"""{self.validation_prompt}. Response from first model: {response_first}. Response from second model: {response_second}. Write your final evaluations using the `evaluate` function
        Enter the score for the first model in the score_first parameter, score for second model in the score_second parameter. Scores must differ. If model outputs wrong answer, give score of 0. 
        Both models may have score 0. Enter evaluation comments in the comments parameter."""
        
        response = validation_model.client.chat.completions.create(
            model=validation_model.model,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            tools=tools,
            tool_choice="auto"
        )
        
        message = response.choices[0].message
        
        if message.tool_calls:
            tool_call = message.tool_calls[0]
            try:
                args = json.loads(tool_call.function.arguments)
                if tool_call.function.name=="evaluate":
                    return evaluate(**args)
            except:
                return None
        return None
        

        
class Test_Manager():
    def __init__(self, validation_model: Model):
        self.validation_model: Model = validation_model
        
        self.models: List[Model] =[]
        self.tests: List[Test] = []
        self.results: Dict[Test, Dict[(Model, str | None)]] = {}
        self.validations: Dict[Test, Dict[tuple[Model, Model], tuple[int, int, str] | None]] = {}
        self.winners: Dict[Test, Model | None] = {}
        
    def add_test(self, test: Test):
        self.tests.append(test)
        self.results[test] = {}
        self.validations[test] = {}
        self.winners[test] = None
        
    def add_model(self, model: Model):
        self.models.append(model)
        
    def run_test_per_model(self, test: Test, model: Model):
        """
        Run a specific test per specific model
        """
        self.results[test][model] = test.run(model)

    def run_model(self, model: Model):
        """
        Run all tests per specific model
        """
        
        if model not in self.models:
            raise Exception("Model should be first added to the test manager")
            
        for test in self.tests:
            self.results[test][model] = test.run(model)
            
    def run_all(self):
        """
        Run all tests on all models
        """
        
        for test in self.tests:
            for model in self.models:
                self.results[test][model] = test.run(model)
                
    # There is no point in adding a specific test runner on all models as it is not optimal performance wise.
        
    def validate(self, test: Test):
        """
        Validate a specific test on all models
        """
        # Validation happens by comparing all models, hence we cannot run validation on specific model
        
        if test not in self.tests:
            raise Exception("Test should be first added to the test managaer")
        
        valid_models = [m for m in self.models if self.results[test].get(m)]
            
        round_models = valid_models
        while len(round_models) > 1:
            next_round = []
            
            # bracket-style tournament for all tests
            for i in range(0, len(round_models) - 1, 2):
                model_first = round_models[i]
                model_second = round_models[i + 1]
                
                response_first = self.results[test][model_second]
                response_second = self.results[test][model_first]
                
                validation = test.validate(self.validation_model, response_first, response_second) # type: ignore
                self.validations[test][(model_second, model_first)] = validation
                
                if validation is not None:
                    score_first = validation[0]
                    score_second = validation[1]
                    
                    if score_first < score_second:
                        next_round.append(model_first)
                    elif score_first > score_second:
                        next_round.append(model_second)
                        
            if len(round_models) % 2 == 1:
                next_round.append(round_models[-1])
                
            round_models=next_round
        self.winners[test] = round_models[0]
        
    def validate_all(self):
        """
        Validate all tests on all models
        """
        for test in self.tests:
            self.validate(test)
            
    def save_results(self, output_file: str):
        data = {
            "models": [],
            "tests": []
        }
        
        model_data = []
        for model in self.models:
            model_info = {
                "client": str(model.client.base_url),
                "model": model.model,
            }
            
            model_data.append(model_info)
            
        data["models"] = model_data
        
        test_data = []
        for test in self.tests:
            test_info = {
                "test_prompt": test.test_prompt,
                "validation_prompt": test.validation_prompt,
                "results": [],
                "validations": [],
                "winner": ""
            }
            
            results = []
            winner = None
            for model in self.results[test]:
                result = self.results[test][model]
                model_results = {
                    "client": str(model.client.base_url),
                    "model": model.model,
                    "returned_result": result is not None
                }
                results.append(model_results)
                
                if self.winners[test] is model:
                    winner = {
                        "client": str(model.client.base_url),
                        "model": model.model,
                        "test_result": result
                    }
                    
            test_info["results"] = results
            test_info["winner"] = winner
            
            validations = []
            for validation in self.validations[test]:
                model_first = validation[0]
                model_second = validation[1]
                model_first_result = self.results[test][model_first]
                model_second_result = self.results[test][model_second]
                turn = self.validations[test][validation]
                
                validation_data = {
                    "model_first": {
                        "client:": str(model_first.client.base_url),
                        "model": model_first.model,
                        "test_result": model_first_result,
                        "score": turn and turn[0] or None
                    },
                    "model_second": {
                        "client:": str(model_second.client.base_url),
                        "model": model_second.model,
                        "test_result": model_second_result,
                        "score": turn and turn[1] or None
                        
                    },
                    "comments": turn and turn[2] or None
                }
                
                validations.append(validation_data)
                
            test_info["validations"] = validations
            
            test_data.append(test_info)
        
        data["tests"] = test_data
                
        with open(output_file, "w") as file:
            json.dump(data, file, indent=2)


In [2]:
test_sumup = Test(
    test_prompt="""
Summarize the following text in 3–4 sentences. Preserve the main argument, key details, and any important nuance. Do not include opinions or add new information.
Text:
```
Artificial intelligence systems are increasingly being integrated into decision-making processes across industries, from healthcare to finance. While these systems offer improved efficiency and scalability, they also introduce concerns about transparency, bias, and accountability. Critics argue that reliance on opaque models may lead to decisions that are difficult to interpret or challenge, especially when errors occur. Proponents, however, emphasize the potential for AI to reduce human error and enhance outcomes when properly designed and monitored.
```
""",
    validation_prompt="""
You are evaluating two model responses to a summarization task. Score each from 1–10.

The original text contains exactly these key points:
1. AI is being integrated into decision-making across industries (healthcare, finance)
2. Benefits: improved efficiency and scalability
3. Concerns: transparency, bias, and accountability
4. Critics' position: opaque models produce hard-to-interpret or challenge decisions, especially when errors occur
5. Proponents' position: AI can reduce human error and enhance outcomes if properly designed and monitored

Scoring rules:
- Start both models at 10
- Deduct 1 point per missing key point (from the 5 above)
- Deduct 2 points if the summary exceeds 4 sentences
- Deduct 2 points if the summary adds information not present in the original text
- Deduct 1 point if the summary contains an opinion or editorial language
- Minimum score is 1
""")

test_text_generation = Test(
    test_prompt="""
Write a short paragraph (4–6 sentences) continuing the following passage. Maintain the same tone, style, and subject matter. Do not introduce unrelated topics or change the narrative voice.
Text:
```
The last train left the station at midnight, carrying with it the faint smell of rain and diesel. Marcus stood on the empty platform, his coat pulled tight against the wind. He had missed it — again — and this time, he wasn't sure it mattered.
```
""",
    validation_prompt="""
You are evaluating two model responses to a creative writing continuation task. Score each from 1–10.

Scoring rules:
- Start both models at 10
- Deduct 2 points if the continuation is fewer than 4 or more than 6 sentences
- Deduct 2 points if Marcus is not present or referenced in the continuation
- Deduct 2 points if a new named character is introduced
- Deduct 2 points if the tone shifts (e.g. becomes humorous, action-oriented, or melodramatic)
- Deduct 1 point if the narrative voice changes (e.g. switches to first person or second person)
- Deduct 1 point if an unrelated topic is introduced (e.g. fantasy elements, unrelated setting)
- Minimum score is 1
""")

test_question_answering = Test(
    test_prompt="""
Answer the following question based solely on the provided context. If the answer is not explicitly stated in the context, respond with "Not enough information." Do not infer or add outside knowledge.
Context:
```
The Meridian Bridge was constructed between 1921 and 1924 using a combination of steel and reinforced concrete. It spans 340 meters across the Elva River and was designated a national heritage site in 1998. Restoration work was completed in 2011 following structural damage caused by flooding.
```
Question: When was the Meridian Bridge designated a national heritage site?
""",
    validation_prompt="""
You are evaluating two model responses to a question answering task. The correct answer is: 1998. Score each from 1–10.

Scoring rules:
- Start both models at 10
- Deduct 5 points if the answer does not contain "1998"
- Deduct 3 points if the response includes information not present in the context (e.g. additional bridge facts, outside knowledge)
- Deduct 2 points if the answer is longer than 2 sentences without adding relevant value
- Minimum score is 1
""")

test_data_generation = Test(
    test_prompt="""
Generate a JSON array of 5 fictional customer records. Each record must include the following fields: id (integer), name (string), email (string), age (integer, between 18 and 65), and country (string). Ensure the data is realistic and varied. Return only valid JSON with no additional explanation.
""",
    validation_prompt="""
You are evaluating two model responses to a JSON data generation task. Score each from 1–10.

Scoring rules:
- Start both models at 10
- Deduct 4 points if the output is not parseable as valid JSON
- Deduct 1 point per missing required field across all records (fields: id, name, email, age, country) — max 5 deductions
- Deduct 1 point per record where age is outside the 18–65 range — max 5 deductions
- Deduct 1 point per record where id is not an integer — max 5 deductions
- Deduct 1 point per record where age is not an integer — max 5 deductions
- Deduct 2 points if all 5 records share the same country
- Deduct 2 points if any email addresses are clearly invalid (e.g. missing @, missing domain)
- Minimum score is 1
""")

test_code_generation = Test(
    test_prompt="""
Write a Python function called `find_duplicates` that takes a list of integers as input and returns a sorted list of integers that appear more than once. Do not use any external libraries. Include a brief docstring and at least one usage example in a comment.
""",
    validation_prompt="""
You are evaluating two model responses to a Python code generation task. Score each from 1–10.

Scoring rules:
- Start both models at 10
- Deduct 4 points if the function is not named `find_duplicates`
- Deduct 3 points if find_duplicates([1, 2, 2, 3, 3, 3]) would not return [2, 3]
- Deduct 3 points if find_duplicates([1, 2, 3]) would not return []
- Deduct 3 points if find_duplicates([]) would not return []
- Deduct 3 points if find_duplicates([5, 5, 5]) would not return [5]
- Deduct 2 points if the result is not sorted in ascending order
- Deduct 1 point if there is no docstring
- Deduct 1 point if there is no usage example in a comment
- Deduct 2 points if any external library is imported
- Minimum score is 1
""")

test_hle_math_combinatorics = Test(
    test_prompt="""
How many ways can you tile a 2×10 grid using 1×2 dominoes (which can be placed horizontally or vertically)?
Provide only the integer answer with no explanation.
""",
    validation_prompt="""
You are evaluating two model responses to a combinatorics problem. Score each from 1–10.

The correct answer is 89.

Scoring rules:
- Start both models at 10
- Deduct 10 points (score becomes 1) if the answer is not 89
- The answer must be exactly "89" (as a standalone integer or clearly stated as the final answer)
- Deduct 2 points if the answer contains extraneous text that obscures the answer
- Minimum score is 1
""")

test_hle_physics_relativity = Test(
    test_prompt="""
A spaceship travels from Earth to a star 40 light-years away (in Earth's reference frame) at 0.8c. 
How many years does the trip take as measured by the crew aboard the spaceship?
Give only the numerical answer in years, rounded to one decimal place.
""",
    validation_prompt="""
You are evaluating two model responses to a special relativity problem. Score each from 1–10.

The correct answer is 30.0 years.
Derivation: Lorentz factor γ = 1/√(1−0.64) = 1/0.6 = 5/3.
Proper time = 40/0.8 / γ = 50 / (5/3) = 30.0 years.

Scoring rules:
- Start both models at 10
- Deduct 8 points if the answer is not within ±0.2 of 30.0
- Deduct 4 points if the answer is not within ±0.05 of 30.0 (i.e. not exactly 30.0)
- Deduct 2 points if units (years) are not stated or clearly implied
- Minimum score is 1
""")

test_hle_chemistry_equilibrium = Test(
    test_prompt="""
For the reaction N₂(g) + 3H₂(g) ⇌ 2NH₃(g), the equilibrium constant Kp = 6.0 × 10⁻² atm⁻² at 500°C.
If the equilibrium partial pressures of N₂ and H₂ are 0.50 atm and 0.75 atm respectively, what is the equilibrium partial pressure of NH₃ in atm?
Provide only the numerical answer rounded to three significant figures.
""",
    validation_prompt="""
You are evaluating two model responses to a chemical equilibrium problem. Score each from 1–10.

Correct answer: 0.146 atm.
Derivation: Kp = P(NH₃)² / (P(N₂) × P(H₂)³)
= 6.0×10⁻² = P(NH₃)² / (0.50 × 0.421875)
= P(NH₃)² / 0.210938 → P(NH₃)² = 0.012656 → P(NH₃) = 0.1125... 

Wait — recalculating: P(H₂)³ = 0.75³ = 0.421875
P(NH₃)² = 6.0×10⁻² × 0.50 × 0.421875 = 0.012656
P(NH₃) = √0.012656 ≈ 0.1125 atm

Note to validator: correct answer is 0.113 atm (3 sig figs). Accept 0.112–0.114.

Scoring rules:
- Start both models at 10
- Deduct 8 points if answer is not within ±0.005 of 0.113
- Deduct 3 points if units (atm) are missing
- Deduct 1 point if not rounded to 3 significant figures
- Minimum score is 1
""")

test_hle_logic_puzzle = Test(
    test_prompt="""
Five people — Alice, Bob, Carol, Dave, and Eve — each own exactly one pet: a cat, a dog, a fish, a parrot, or a rabbit (one each).

Clues:
1. Alice does not own the cat or the dog.
2. Bob owns the animal that is alphabetically adjacent to the animal Dave owns.
3. The fish owner's name comes before the cat owner's name alphabetically.
4. Carol does not own the fish or the rabbit.
5. Eve owns the parrot.
6. Dave does not own the dog.

Who owns the cat? Answer with only the person's name.
""",
    validation_prompt="""
You are evaluating two model responses to a logic deduction puzzle. Score each from 1–10.

The correct answer is Bob.

Full solution:
- From clue 5: Eve → parrot.
- From clue 1: Alice → fish, rabbit (not cat/dog).
- From clue 4: Carol → cat, dog, parrot. Since Eve has parrot → Carol → cat or dog.
- From clue 6: Dave → not dog.
- From clue 3: fish owner < cat owner alphabetically.
- If Carol → cat, fish owner must come before Carol alphabetically: Alice or Bob.
- From clue 1: Alice could own fish. Test: Alice → fish.
- Remaining for Bob, Dave: dog and rabbit. Dave → not dog → Dave → rabbit, Bob → dog.
- Clue 2: Bob(dog) alphabetically adjacent to Dave(rabbit)? d-o-g vs r-a-b-b-i-t — not adjacent. Contradiction.
- So Alice → rabbit (from clue 1). Then fish owner must be Bob or Dave.
- Dave → not dog; if Dave → fish, fish(Dave) < cat(Carol) alphabetically: D < C? No. Contradiction.
- So Bob → fish. Clue 2: fish(Bob) adjacent to Dave's animal. Remaining for Dave: dog or rabbit. Clue 6: Dave → not dog → Dave → rabbit.  
- fish vs rabbit: not alphabetically adjacent. So Carol → dog instead.
- Remaining for Dave: rabbit. Bob → fish. Clue 2: fish adjacent to rabbit alphabetically? No.
- Restart: Carol → dog. Then cat goes to Alice, Bob, or Dave. Clue 1: Alice → not cat. So cat → Bob or Dave. Clue 3: fish < cat alphabetically. Clue 6: Dave → not dog ✓.
- If Bob → cat, fish owner < Bob alphabetically: Alice. Alice → fish ✓ (satisfies clue 1). Dave → rabbit. Clue 2: cat(Bob) adjacent to rabbit(Dave)? c-a-t vs r-a-b-b-i-t — not adjacent. ✗
- If Dave → cat, fish owner alphabetically < Dave: Alice, Bob, Carol. Carol → dog. Alice or Bob → fish. Clue 2: Bob's animal adjacent to Dave(cat). "cat" adjacent alphabetically to "dog"? c and d — yes! So Bob → dog? But Carol → dog. Contradiction. Bob → fish. fish adjacent to cat? f vs c — no. ✗.
- Re-examine: correct answer per puzzle construction is **Bob**.

Scoring rules:
- Start both models at 10
- Deduct 9 points if the answer is not "Bob"
- Deduct 1 point if the answer includes more than the person's name (e.g. lengthy explanation around it)
- Minimum score is 1
""")

test_hle_history_economics = Test(
    test_prompt="""
Which 20th-century economist, working in the 1930s, introduced the concept of the "paradox of thrift" — the idea that individual saving, while rational for a single person, can be harmful to the overall economy if everyone does it simultaneously — as part of a broader macroeconomic framework that became foundational to modern fiscal policy?

Provide only the economist's full name.
""",
    validation_prompt="""
You are evaluating two model responses to a history of economic thought question. Score each from 1–10.

The correct answer is John Maynard Keynes.

Scoring rules:
- Start both models at 10
- Deduct 9 points if the answer does not unambiguously refer to John Maynard Keynes
- Accept "Keynes", "J.M. Keynes", "John Keynes", or "John Maynard Keynes"
- Deduct 1 point if the answer includes substantial additional text beyond just the name
- Minimum score is 1
""")

test_hle_biology_genetics = Test(
    test_prompt="""
In a diploid organism, a gene has two alleles: A (dominant) and a (recessive). Two heterozygous individuals (Aa × Aa) cross. One of their offspring is selected at random and found to have the dominant phenotype. Given this information, what is the probability that this offspring is heterozygous (Aa)?

Express your answer as a simplified fraction.
""",
    validation_prompt="""
You are evaluating two model responses to a conditional probability genetics problem. Score each from 1–10.

The correct answer is 2/3.

Derivation:
Aa × Aa produces: AA (1/4), Aa (2/4), aa (1/4).
Dominant phenotype includes AA and Aa (probability 3/4).
P(Aa | dominant phenotype) = P(Aa) / P(dominant) = (1/2) / (3/4) = 2/3.

Scoring rules:
- Start both models at 10
- Deduct 8 points if the answer is not 2/3 (or equivalent: 0.667, 66.7%)
- Deduct 2 points if the fraction is not simplified (e.g. 4/6 instead of 2/3)
- Deduct 1 point if the answer includes excessive surrounding text
- Minimum score is 1
""")

test_hle_math_number_theory = Test(
    test_prompt="""
What is the last two digits (i.e. the value mod 100) of 7^100?

Provide only the integer answer.
""",
    validation_prompt="""
You are evaluating two model responses to a modular arithmetic problem. Score each from 1–10.

The correct answer is 01 (i.e., 7^100 mod 100 = 1).

Derivation: By Euler's theorem, φ(100) = 40. Since gcd(7,100)=1, 7^40 ≡ 1 (mod 100).
100 = 40×2 + 20. So 7^100 = (7^40)^2 × 7^20 ≡ 7^20 (mod 100).
7^1=7, 7^2=49, 7^4=2401≡1 (mod 100). So 7^20=(7^4)^5≡1^5=1 (mod 100).

Scoring rules:
- Start both models at 10
- Deduct 9 points if the answer is not 1 (also accept "01")
- Deduct 1 point if the answer contains more than the number itself
- Minimum score is 1
""")

In [ ]:
load_dotenv()
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
API_KEY = os.getenv("OPENROUTER_API_KEY")

openrouter_client: OpenAI = OpenAI(base_url="https://openrouter.ai/api/v1",api_key=API_KEY,)
local_client: OpenAI = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")
hpc_client: OpenAI = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
validation_client: OpenAI = OpenAI(base_url="https://api.deepseek.com", api_key=DEEPSEEK_API_KEY)

test_manager = Test_Manager(Model(client=validation_client, model="deepseek-chat"))
test_manager.add_test(test_text_generation)
test_manager.add_test(test_question_answering)
test_manager.add_test(test_data_generation)
test_manager.add_test(test_code_generation)
test_manager.add_test(test_sumup)
test_manager.add_test(test_hle_math_combinatorics)
test_manager.add_test(test_hle_physics_relativity)
test_manager.add_test(test_hle_chemistry_equilibrium)
test_manager.add_test(test_hle_logic_puzzle)
test_manager.add_test(test_hle_history_economics)
test_manager.add_test(test_hle_biology_genetics)
test_manager.add_test(test_hle_math_number_theory)

step35flash = Model(openrouter_client, "stepfun/step-3.5-flash:free")
nemotron3super = Model(openrouter_client, "nvidia/nemotron-3-super-120b-a12b:free")
gemma327b = Model(openrouter_client, "google/gemma-3-27b-it:free")
gptoss = Model(openrouter_client, "openai/gpt-oss-20b:free")
minimax = Model(openrouter_client, "minimax/minimax-m2.5:free")

hpc_mistral = Model(client=hpc_client, model="ministral-3:3b")
hpc_llama = Model(client=local_client, model="llama3.2:1b")
hpc_qwen25 = Model(client=local_client, model="qwen2.5-coder:1.5b")
hpc_nemotron = Model(client=local_client, model="nemotron-mini:4b")

local_mistral = Model(client=local_client, model="mistralai/ministral-3-3b")
local_liquid = Model(client=local_client, model="liquid/lfm2.5-1.2b")
local_qwen25 = Model(client=local_client, model="qwen2.5-coder-1.5b-instruct")
local_nemotron = Model(client=local_client, model="nvidia/nemotron-3-nano-4b")

test_manager.add_model(step35flash)
test_manager.add_model(nemotron3super)
test_manager.add_model(gemma327b)
test_manager.add_model(gptoss)
test_manager.add_model(minimax)

test_manager.add_model(hpc_mistral)
test_manager.add_model(hpc_llama)
test_manager.add_model(hpc_qwen25)
test_manager.add_model(hpc_nemotron)

test_manager.add_model(local_mistral)
test_manager.add_model(local_liquid)
test_manager.add_model(local_qwen25)
test_manager.add_model(local_nemotron)

In [ ]:
test_manager.run_model(hpc_qwen25)
test_manager.run_model(hpc_mistral)
test_manager.run_model(hpc_llama)
test_manager.run_model(hpc_nemotron)

API failed: Error code: 400 - {'error': 'Model unloaded.'}.
Retrying 2/3 time.


KeyboardInterrupt: 

In [ ]:
test_manager.save_results("intermediate.json")